In [1]:
from chewbacca.datamodules.components.kubric_dataset import create_point_tracking_dataset

2025-01-13 01:00:44.006088: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
tf_dataset = create_point_tracking_dataset(
              train_size=(112, 112),
              shuffle_buffer_size=2,
              split="train",
              repeat=True,
              vflip=False,
              random_crop=True,
              tracks_to_sample=256,
          )

2025-01-13 01:01:38.354667: W external/local_tsl/tsl/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Couldn't resolve host name', error details: Could not resolve host: metadata.google.internal".


Instructions for updating:
`seed2` arg is deprecated.Use sample_distorted_bounding_box_v2 instead.


Instructions for updating:
`seed2` arg is deprecated.Use sample_distorted_bounding_box_v2 instead.


Instructions for updating:
Use `tf.random.categorical` instead.


Instructions for updating:
Use `tf.random.categorical` instead.


In [5]:
import tensorflow_datasets as tfds
generator = tfds.as_numpy(tf_dataset)

In [42]:
import numpy as np
import torch
def process_batch(sample):
    video_np = sample["video"]               # shape: (24, 256, 256, 3)
    query_points_np = sample["query_points"] # shape: (256, 3)
    target_points_np = sample["target_points"] # shape: (256, 24, 2)
    

    # Convert to torch
    
    video_np = video_np * .5 + .5
    # video_np = (video_np - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
    video_torch = torch.from_numpy(video_np)

    # query_points_np[:, 1:] /= 112
    # target_points_np /= 112
    
    query_points_torch = torch.from_numpy(query_points_np)
    target_points_torch = torch.from_numpy(target_points_np)

    # If you prefer (C, T, H, W) for the video, you might permute:
    video_torch = video_torch.permute(3, 0, 1, 2) 

    return video_torch.numpy(), query_points_torch.numpy(), target_points_torch.numpy()

In [43]:
from chewbacca.utils import tapvid_viz_utils

# gen_iter = iter(generator)

In [44]:
sample = next(gen_iter)
video, query_points, target_points = process_batch(sample)

In [60]:
norm_points = (target_points) / 256

In [68]:
scale_factor = np.array([112, 112])[np.newaxis, np.newaxis, :]

painted_frames_pred = tapvid_viz_utils.paint_point_track(
                    video.transpose(1, 2, 3, 0) * 255,
                    norm_points * 112,
                    ~sample["occluded"]
                )

In [69]:
import imageio; 
from IPython.display import Video; 
imageio.mimwrite('test2.mp4', painted_frames_pred, fps=2); 
Video('test2.mp4', width=480, height=360) #the width and height option as additional thing new in Ipython 7.6.1

In [65]:
sample["occluded"].astype(

array([[False, False, False, ..., False, False, False],
       [False, False, False, ...,  True,  True,  True],
       [False, False, False, ..., False, False, False],
       ...,
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False]])